# Step 2 — Gmsh meshing

**Learning objective:** Generate the 2D triangular mesh and inspect refinement near the wall toe.

**Run from top to bottom in VS Code.** Each notebook executes one workshop step and shows its output. Select the working OGS Postprocessing Python kernel.

**Prerequisites:** run earlier numbered steps. `GMSH2OGS` and `ogs` must be on PATH when used; Gmsh is imported from the selected Python kernel in Step 2. **Every image below represents a distinct quantity.** Green PASS messages verify the stated condition only.

**Environment note:** This notebook calls the Gmsh Python API from the active VS Code kernel. It does not invoke `/usr/local/bin/gmsh`, which may point to a wrapper using a different Python installation. Install `gmsh` into this kernel environment if missing.


In [ ]:
from pathlib import Path
import sys, subprocess
import numpy as np
import matplotlib.pyplot as plt
import meshio
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents, Path.cwd().parent) if (p / "models" / "excavation.prj").exists()), None)
assert ROOT is not None, "Open the extracted project directory in VS Code before running this notebook"
print('Training root:', ROOT)
print('Python:', sys.executable)
def run(*command):
    command = [str(c) for c in command]
    print('RUN:', ' '.join(command), flush=True)
    subprocess.run(command, cwd=ROOT, check=True)


**Run:** generate `mesh/excavation.msh` from the original `.geo` file via the Gmsh Python API using MSH 2.1. Then check the mesh before plotting. The terminal launcher `/usr/local/bin/gmsh` is deliberately bypassed.

In [ ]:
# The executable /usr/local/bin/gmsh may use a different Python installation.
# Import from the exact interpreter used by this VS Code notebook instead.
try:
    import gmsh
except ImportError as exc:
    raise RuntimeError(
        'Gmsh is not available in this notebook kernel. In the TEC shell run:\n'
        '  ~/.local/bin/uv pip install --python /home/simoe/pip_apps/.venv-ogs/bin/python gmsh\n'
        'Then restart this VS Code kernel and rerun the notebook.'
    ) from exc

mesh_path = ROOT / 'mesh' / 'excavation.msh'
mesh_path.parent.mkdir(exist_ok=True)
gmsh.initialize()
try:
    gmsh.option.setNumber('General.Terminal', 1)
    gmsh.open(str(ROOT / 'geometry' / 'excavation.geo'))
    gmsh.option.setNumber('Mesh.MshFileVersion', 2.1)
    gmsh.model.mesh.generate(2)
    gmsh.write(str(mesh_path))
finally:
    gmsh.finalize()

assert mesh_path.exists() and mesh_path.stat().st_size > 0
print('Created Gmsh mesh:', mesh_path)
run(sys.executable, 'scripts/check_wall_mesh.py', 'mesh/excavation.msh')

**Inspect:** show the actual mesh produced by Gmsh.

In [ ]:
m=meshio.read(ROOT/'mesh'/'excavation.msh')
tris=np.vstack([b.data for b in m.cells if b.type=='triangle'])
fig,ax=plt.subplots(figsize=(12,5.6))
ax.triplot(m.points[:,0],m.points[:,1],tris,color='#2563eb',lw=.45)
ax.set(xlim=(0,65),ylim=(0,30),xlabel='x [m]',ylabel='y [m]',title=f'Gmsh mesh: {len(m.points)} nodes / {len(tris)} triangles')
ax.set_aspect('equal');plt.show()
print('PASS: wall opening and mesh preview available')

**Interpretation:** The thin triangular lines show individual finite elements; this is **mesh topology**, not a physical field. Look for smaller triangles around the wall tip at (32.5 m, 8 m). The wall gap contains no triangles. **PASS:** `check_wall_mesh.py` verifies the notch vertices and mesh structure. **Next:** convert to OGS VTU and inspect MaterialIDs.